# Exercise 1: Exploring Different Optimizers in Logistic Regression

## Objective
In this exercise, you will explore different types of optimizers (e.g., LBFGS, SAG) in logistic regression using multi-dimensional data with features at different scales. You'll observe how different algorithms perform on this dataset without any preprocessing.

## Dataset
We'll create a synthetic dataset with the following characteristics:
- Multiple dimensions
- Features at different scales
- At least one exponentially distributed feature
- At least one categorical feature

## Tasks

1. Generate the dataset as described above.
2. Use an OrdinalEncoder for the categorical data.
3. Split the data into training and test sets.
4. Implement logistic regression with different optimizers:
   - LBFGS
   - SAGA
   - SAG (Stochastic Average Gradient)
5. Compare the performance of each optimizer.

## Starter Code


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

# Generate synthetic data
np.random.seed(42)
n_samples = 10000

# Numeric features
feature1 = np.random.normal(0, 1, n_samples)
feature2 = np.random.exponential(2, n_samples)
feature3 = np.random.uniform(0, 10000, n_samples)

# Categorical feature
categories = ['A', 'B', 'C', 'D']
feature4 = np.random.choice(categories, n_samples)

# Target: a rule combining all four features
y = (0.2 * feature1 + np.log(feature2) + 0.0001 * feature3 + np.where(feature4 == 'A', 1, 0) > 2).astype(int)

# Create a DataFrame
df = pd.DataFrame({'feature1': feature1, 'feature2': feature2,
                   'feature3': feature3, 'feature4': feature4})
df['target'] = y

# Encode categorical feature
encoder = OrdinalEncoder()
df['feature4'] = encoder.fit_transform(df[['feature4']])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(df.drop('target', axis=1), df['target'], test_size=0.2, random_state=42)

# TODO: Implement logistic regression with different optimizers
# Hint: Use LogisticRegression(solver='lbfgs'), LogisticRegression(solver='saga'), and LogisticRegression(solver='sag')


# TODO: Compare the performance of each optimizer
# Hint: Use accuracy_score and log_loss to evaluate performance


## Questions

1. Which optimizer performs best on this dataset? Why do you think that is?
   *(Hint: about 80% of the rows are class 0. What accuracy would you get by always predicting 0?)*
2. How does the performance vary between the training and test sets for each optimizer?
3. Do you notice any issues with convergence for any of the optimizers?
4. How might the different scales of the features be affecting the performance of each optimizer?

_Answer here_

# Exercise 2: Preprocessing in a Pipeline

## Objective
In Exercise 1 the optimizers saw raw features on wildly different scales (`feature3` runs to 10,000; `feature1` is around 0). Here you'll add preprocessing and see how the same optimizers behave once the features are on comparable scales.

## A new tool: `ColumnTransformer` (given)
Our data mixes numeric and categorical columns, and they need *different* preprocessing: the numeric columns should be scaled, and the categorical column should be one-hot encoded. `ColumnTransformer` does exactly this. You give it a list of `(name, transformer, columns)` entries; it applies each transformer to its own columns and puts the results side by side. It behaves like any other transformer, so it can be the first step of a `Pipeline`.

**We've written the `ColumnTransformer` for you.** You don't need to master it this week (we study preprocessing pipelines properly in week 9). Your job is to *use* it.

## Tasks
1. Run the setup cell below. It regenerates **the same data and the same train/test split** as Exercise 1, and defines `preprocessor`.
2. Build a `Pipeline` of `preprocessor` followed by `LogisticRegression`, and train it with each solver (`lbfgs`, `sag`, `saga`), using `max_iter=1000`. Report test accuracy and log loss, and compare them with your Exercise 1 results.
3. Make **one** change to the preprocessing and see what happens. Pick either:
   - swap `StandardScaler` for `MinMaxScaler`, or
   - log-transform `feature2` before scaling it, using the `LogTransform` class you wrote in the week-6 lab (copy it in), or `FunctionTransformer(np.log)`, which does the same thing.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

# Same data as Exercise 1 (same seed, same distributions, same split)
np.random.seed(42)
n_samples = 10000
feature1 = np.random.normal(0, 1, n_samples)
feature2 = np.random.exponential(2, n_samples)
feature3 = np.random.uniform(0, 10000, n_samples)
feature4 = np.random.choice(['A', 'B', 'C', 'D'], n_samples)
y = (0.2 * feature1 + np.log(feature2) + 0.0001 * feature3 + np.where(feature4 == 'A', 1, 0) > 2).astype(int)

df = pd.DataFrame({'feature1': feature1, 'feature2': feature2,
                   'feature3': feature3, 'feature4': feature4})
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42)

# GIVEN: the preprocessing step. Each entry is (name, transformer, columns it applies to).
numeric_features = ['feature1', 'feature2', 'feature3']
categorical_features = ['feature4']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),     # scale the numeric columns
    ('cat', OneHotEncoder(), categorical_features),  # one-hot encode the categorical column
])

# Peek at what it produces: 3 scaled numeric columns, then 4 one-hot columns (A, B, C, D)
preprocessor.fit_transform(X_train)[:3].round(2)

In [ ]:
# TODO (Task 2): put `preprocessor` and LogisticRegression together in a Pipeline,
# train one pipeline per solver ('lbfgs', 'sag', 'saga') with max_iter=1000,
# and print test accuracy and log loss for each.
# Hint: log_loss needs predicted probabilities, so use pipeline.predict_proba(X_test)

In [ ]:
# TODO (Task 3): make ONE change to the preprocessing, rebuild the pipeline, and compare.
#
# Option A: replace StandardScaler() with MinMaxScaler() in a copy of `preprocessor`.
#
# Option B: give feature2 its own entry that logs it and then scales it, and remove
# 'feature2' from the 'num' list. For example:
#   ('log_f2', Pipeline([('log', LogTransform()), ('scale', StandardScaler())]), ['feature2'])
# (FunctionTransformer(np.log) works in place of LogTransform().)

## Questions

1. How did preprocessing change the results for each optimizer compared with Exercise 1? Which optimizer changed the most? Thinking back to gradient descent in week 6, why would feature scale matter so much to it?
2. What change did you make in Task 3, and what happened? If you tried the log transform, look at how `y` is generated. Why might it help so much?
3. In Exercise 1, `lbfgs` did reasonably well even without scaling. Does that mean scaling is unnecessary when you use `lbfgs`? Explain.

_Answer here_